In [ ]:
# 2-DOF companion SPEC §8: PE8 (P5a torque), PE3 (P2) and PE4 (P3), seeds 2-4, residual SAC and SAC+HER.
# Seeds 0-1 run on the laptop (results/runs/pe34_offloaded_s234, pe8_offloaded_s234). Jobs run in
# parallel, one per CPU core, 1 torch thread each (a CPU or TPU VM; no accelerator is used).
# Needs ur5rl_bundle.zip built 2026-10-29 or later. CPU runtime. Jobs, in this order:
#   1. residual_787d568a_s2
#   2. sac_her_db65e789_s2
#   3. residual_787d568a_s3
#   4. sac_her_db65e789_s3
#   5. residual_787d568a_s4
#   6. sac_her_db65e789_s4
#   7. residual_b1a95724_s2
#   8. sac_her_55c2c989_s2
#   9. residual_b1a95724_s3
#   10. sac_her_55c2c989_s3
#   11. residual_b1a95724_s4
#   12. sac_her_55c2c989_s4
#   13. residual_dd445192_s2
#   14. sac_her_863c2971_s2
#   15. residual_dd445192_s3
#   16. sac_her_863c2971_s3
#   17. residual_dd445192_s4
#   18. sac_her_863c2971_s4
# Colab: bundle at MyDrive/mtp/ur5rl_bundle.zip. Checkpoints (model + replay buffer, ~0.85 GB per run) go to
#   MyDrive/mtp/runs every 50k steps (needs ~16.7 GB free on Drive). After a disconnect, run this cell again: it
#   resumes (<= 50k steps lost). When done, copy MyDrive/mtp/runs/pe8|pe3|pe4/<run id> into results/runs/pe8|pe3|pe4/ on the laptop.
# Kaggle: attach the bundle as a dataset, Internet on, run with "Save & Run All (Commit)". Runs go to this version's
#   output (runs/pe8|pe3|pe4/<run id>); training stops before the 12 h limit. To continue, attach that output as an input
#   of a new version and commit again: it resumes from it.
JOBS = [('2-dof/configs/exp/pe8.yaml', 'residual', 2), ('2-dof/configs/exp/pe8.yaml', 'sac_her', 2), ('2-dof/configs/exp/pe8.yaml', 'residual', 3), ('2-dof/configs/exp/pe8.yaml', 'sac_her', 3), ('2-dof/configs/exp/pe8.yaml', 'residual', 4), ('2-dof/configs/exp/pe8.yaml', 'sac_her', 4), ('2-dof/configs/exp/pe3.yaml', 'residual', 2), ('2-dof/configs/exp/pe3.yaml', 'sac_her', 2), ('2-dof/configs/exp/pe3.yaml', 'residual', 3), ('2-dof/configs/exp/pe3.yaml', 'sac_her', 3), ('2-dof/configs/exp/pe3.yaml', 'residual', 4), ('2-dof/configs/exp/pe3.yaml', 'sac_her', 4), ('2-dof/configs/exp/pe4.yaml', 'residual', 2), ('2-dof/configs/exp/pe4.yaml', 'sac_her', 2), ('2-dof/configs/exp/pe4.yaml', 'residual', 3), ('2-dof/configs/exp/pe4.yaml', 'sac_her', 3), ('2-dof/configs/exp/pe4.yaml', 'residual', 4), ('2-dof/configs/exp/pe4.yaml', 'sac_her', 4)]
import glob, os, shutil, subprocess, time
T0 = time.time()
sh = lambda c: subprocess.run(c, shell=True, check=True)
if os.path.exists('/kaggle'):
    W, OUT, DEADLINE = '/tmp/ur5rl', '/kaggle/working/runs', T0 + 11.4 * 3600  # Kaggle stops a session at 12 h
    zips = glob.glob('/kaggle/input/**/ur5rl_bundle.zip', recursive=True)
    if zips:
        sh(f'mkdir -p {W} && unzip -q -o {zips[0]} -d {W}')
    else:  # Kaggle may unpack an uploaded zip into a folder
        root = os.path.dirname(os.path.dirname(glob.glob('/kaggle/input/**/scripts/train.py', recursive=True)[0]))
        shutil.copytree(root, W, dirs_exist_ok=True)
    for prev in glob.glob('/kaggle/input/**/runs/*/*_s[0-9]', recursive=True):  # an earlier version's output: resume
        dst = os.path.join(OUT, *prev.rstrip('/').split('/')[-2:])
        if not os.path.exists(dst):
            shutil.copytree(prev, dst)
            print('resuming from', prev)
else:
    from google.colab import drive
    drive.mount('/content/drive')
    W, OUT, DEADLINE = '/content/ur5rl', '/content/drive/MyDrive/mtp/runs', None
    sh(f'mkdir -p {W} && unzip -q -o /content/drive/MyDrive/mtp/ur5rl_bundle.zip -d {W}')  # every time: current code
if not os.path.exists('/tmp/.deps_ok'):  # fresh VM: install once
    sh('pip install -q --index-url https://download.pytorch.org/whl/cpu torch==2.14.1')
    sh('pip install -q mujoco==3.17.0 gymnasium==1.4.0 stable-baselines3==2.9.2 numpy==2.4.6 scipy==1.17.1 '
       'PyYAML==6.0.3 psutil==7.2.2 tensorboard==2.21.0')
    sh(f'pip install -q -e {W} --no-deps --ignore-requires-python')
    if os.path.exists(f'{W}/2-dof/pyproject.toml'):
        sh(f'pip install -q -e {W}/2-dof --no-deps --ignore-requires-python')
    sh('touch /tmp/.deps_ok')
WORKERS = 0
THREADS = 2 if WORKERS == 1 else 1


def job(j):
    exp, run, seed = j
    left = None if DEADLINE is None else int(DEADLINE - time.time())
    if left is not None and left < 900:
        print(f'skip {run} s{seed}: too close to the Kaggle time limit (commit a new version with this output attached)')
        return
    print(f'=== {exp} {run} seed {seed}', flush=True)
    limit = f'timeout {left}s ' if left else ''
    subprocess.run(f'cd {W} && {limit}python scripts/train.py --exp {exp} --run {run} --seed {seed} --threads {THREADS} '
                   f'--out {OUT} --ckpt-every 50000 --keep-full-at 2>&1 '
                   f'| grep --line-buffered -E "val]|resumed|done|Error|Traceback" | sed -u "s/^/[{run} s{seed}] /"',
                   shell=True, check=DEADLINE is None)  # Kaggle: a run stopped by the time limit must not fail the cell


if WORKERS == 1:
    for j in JOBS:
        job(j)
else:
    from concurrent.futures import ThreadPoolExecutor
    n = max(1, min(len(JOBS), (os.cpu_count() or 2)))
    print(f'{os.cpu_count()} CPUs: {n} jobs in parallel', flush=True)
    with ThreadPoolExecutor(n) as pool:
        list(pool.map(job, JOBS))
print('finished (or stopped at the time limit); finished runs contain done.json')
